In [0]:
# ============================================
# DEFINE ALL WIDGETS (ADF REQUIRED)
# ============================================

# Remove old widgets first to avoid conflicts
for w in ["env", "run_date", "storage_account", "raw_container", "bronze_container"]:
    try:
        dbutils.widgets.remove(w)
    except:
        pass

# Define widgets with defaults (ADF will override these)
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("run_date", "2018-03-04")
dbutils.widgets.text("storage_account", "vasudatalake1")
dbutils.widgets.text("raw_container", "raw")
dbutils.widgets.text("bronze_container", "bronze")

print("✅ All widgets initialized")

# ============================================
# GET WIDGET VALUES
# ============================================

env             = dbutils.widgets.get("env")
run_date        = dbutils.widgets.get("run_date")
storage_account = dbutils.widgets.get("storage_account")
raw_container   = dbutils.widgets.get("raw_container")
bronze_container= dbutils.widgets.get("bronze_container")

print(f"env={env}, run_date={run_date}, storage_account={storage_account}")

# ============================================
# SPLIT DATE INTO YEAR/MONTH/DAY
# ============================================

year, month, day = run_date.split("-")

# ============================================
# BUILD PATHS
# ============================================

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<storage_key>"
)

raw_path    = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/{env}/{year}/{month}/{day}/"
bronze_path = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/{env}/{year}/{month}/{day}/"

print("✅ Raw Path:", raw_path)
print("✅ Bronze Path:", bronze_path)

# ============================================
# VERIFY PATH EXISTS BEFORE PROCEEDING
# ============================================

try:
    files = dbutils.fs.ls(raw_path)
    print(f"✅ Path verified! Found {len(files)} file(s)")
except Exception as e:
    raise Exception(f"❌ Path not found: {raw_path}\nCheck that data exists for run_date={run_date}")

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load(raw_path)

display(df)

In [0]:
print("Row count:", df.count())

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn("ingestion_time", current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)
display(bronze_df)

In [0]:
if df.count() == 0:
    raise Exception("No data found in raw layer")

In [0]:
display(dbutils.fs.ls(raw_path))

In [0]:
df = spark.read.csv(raw_path, header=True)
print("Rows in RAW:", df.count())

In [0]:
# ============================================
# 📥 STEP 5: READ MULTIPLE TABLES
# ============================================

orders_df   = spark.read.csv(raw_path + "orders_*.csv",         header=True, inferSchema=True)
payments_df = spark.read.csv(raw_path + "order_payments_*.csv", header=True, inferSchema=True)
items_df    = spark.read.csv(raw_path + "order_items_*.csv",    header=True, inferSchema=True)
reviews_df  = spark.read.csv(raw_path + "order_reviews_*.csv",  header=True, inferSchema=True)

print("📊 Orders:",   orders_df.count())
print("📊 Payments:", payments_df.count())
print("📊 Items:",    items_df.count())
print("📊 Reviews:",  reviews_df.count())

# ============================================
# 🥉 STEP 6: WRITE BRONZE (DELTA)
# ============================================

# Disable format check + wipe entire bronze date folder for clean write
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")

dbutils.fs.rm(bronze_path, recurse=True)
print(f"✅ Cleared bronze path: {bronze_path}")

orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_path + "orders/")
payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_path + "payments/")
items_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_path + "items/")
reviews_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_path + "reviews/")

print("✅ Bronze Layer Completed")

# ============================================
# 🥈 STEP 7: SILVER (JOIN + CLEAN)
# ============================================

from pyspark.sql.functions import col

orders_bronze   = spark.read.format("delta").load(bronze_path + "orders/")
payments_bronze = spark.read.format("delta").load(bronze_path + "payments/")
items_bronze    = spark.read.format("delta").load(bronze_path + "items/")

# Drop columns that would cause duplicates in join
payments_bronze = payments_bronze.drop("ingestion_time")
items_bronze    = items_bronze.drop("ingestion_time")

# Join all three
silver_df = orders_bronze \
    .join(payments_bronze, "order_id", "left") \
    .join(items_bronze,    "order_id", "left") \
    .filter(col("order_id").isNotNull())

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/{env}/orders/"

silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print("✅ Silver Layer Completed")

# ============================================
# 🥇 STEP 8: GOLD (BUSINESS METRIC)
# ============================================

from pyspark.sql.functions import sum as spark_sum  # avoid conflict with Python built-in sum

gold_df = silver_df.groupBy("order_status") \
    .agg(spark_sum("payment_value").alias("total_revenue"))

gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/{env}/metrics/"

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_path)

print("✅ Gold Layer Completed")

display(gold_df)

print("🎉 FULL PIPELINE SUCCESS")

In [0]:
from pyspark.sql.functions import current_timestamp

print("🚀 Starting Bronze Layer...")

# -----------------------------
# PARAMETERS (ADF or fallback)
# -----------------------------
try:
    env = dbutils.widgets.get("env")
    run_date = dbutils.widgets.get("run_date")
except:
    env = "dev"
    run_date = "2018-03-04"

print(f"📌 env={env}, run_date={run_date}")

# -----------------------------
# DATE SPLIT (DYNAMIC)
# -----------------------------
year, month, day = run_date.split("-")

print(f"📅 Partition → {year}/{month}/{day}")

# -----------------------------
# STORAGE CONFIG
# -----------------------------
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<storage_key>"
)

print("🔐 Storage Connected")

# -----------------------------
# PATHS
# -----------------------------
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/{env}/{year}/{month}/{day}/"

orders_path   = base_raw + "orders_*.csv"
payments_path = base_raw + "order_payments_*.csv"
items_path    = base_raw + "order_items_*.csv"
reviews_path  = base_raw + "order_reviews_*.csv"

bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net/{env}/"

print("📂 Raw Base:", base_raw)

# -----------------------------
# READ FILES
# -----------------------------
orders_df = spark.read.option("header", True).option("inferSchema", True).csv(orders_path)
payments_df = spark.read.option("header", True).option("inferSchema", True).csv(payments_path)
items_df = spark.read.option("header", True).option("inferSchema", True).csv(items_path)
reviews_df = spark.read.option("header", True).option("inferSchema", True).csv(reviews_path)

print("📊 Orders:", orders_df.count())
print("📊 Payments:", payments_df.count())
print("📊 Items:", items_df.count())
print("📊 Reviews:", reviews_df.count())

# -----------------------------
# ADD INGESTION TIME
# -----------------------------
orders_df = orders_df.withColumn("ingestion_time", current_timestamp())
payments_df = payments_df.withColumn("ingestion_time", current_timestamp())
items_df = items_df.withColumn("ingestion_time", current_timestamp())
reviews_df = reviews_df.withColumn("ingestion_time", current_timestamp())

print("🧹 Added ingestion_time")

# -----------------------------
# WRITE BRONZE (DELTA)
# -----------------------------
orders_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(bronze_base + "orders/")
payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_base + "payments/")
items_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_base + "items/")
reviews_df.write.format("delta").mode("overwrite").save(bronze_base + "reviews/")

print("✅ Bronze Layer Completed")

In [0]:
display(dbutils.fs.ls(bronze_base))

In [0]:
display(dbutils.fs.ls("abfss://bronze@vasudatalake1.dfs.core.windows.net/dev/"))

In [0]:
from pyspark.sql.functions import col, to_timestamp

print("🚀 Starting Silver Layer...")

# -----------------------------
# PATHS
# -----------------------------
bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net/{env}/"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net/{env}/"

# -----------------------------
# READ BRONZE
# -----------------------------
orders_df   = spark.read.format("delta").load(bronze_base + "orders/")
payments_df = spark.read.format("delta").load(bronze_base + "payments/")
items_df    = spark.read.format("delta").load(bronze_base + "items/")
reviews_df  = spark.read.format("delta").load(bronze_base + "reviews/")

print("📊 Bronze Rows:")
print("Orders:", orders_df.count())
print("Payments:", payments_df.count())
print("Items:", items_df.count())
print("Reviews:", reviews_df.count())

# -----------------------------
# CLEANING
# -----------------------------
orders_df = orders_df.dropDuplicates(["order_id"])

# Fix timestamps
orders_df = orders_df \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at"))

print("🧹 Cleaned & fixed timestamps")

# -----------------------------
# REMOVE DUPLICATE COLUMNS
# -----------------------------
payments_df = payments_df.drop("ingestion_time")
items_df    = items_df.drop("ingestion_time")
reviews_df  = reviews_df.drop("ingestion_time")

print("🧹 Removed duplicate ingestion_time columns")

# -----------------------------
# JOIN TABLES
# -----------------------------
silver_df = orders_df \
    .join(payments_df, "order_id", "left") \
    .join(items_df, "order_id", "left") \
    .join(reviews_df, "order_id", "left")

print("🔗 Joined all tables")

# -----------------------------
# WRITE SILVER
# -----------------------------
silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(silver_base + "ecommerce/")

print("✅ Silver Layer Completed")

display(silver_df)

In [0]:
from pyspark.sql.functions import sum, count, avg

print("🚀 Starting Gold Layer...")

# -----------------------------
# PATHS
# -----------------------------
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net/{env}/"
gold_base   = f"abfss://gold@{storage_account}.dfs.core.windows.net/{env}/"

# -----------------------------
# READ SILVER
# -----------------------------
silver_df = spark.read.format("delta") \
    .load(silver_base + "ecommerce/")

print("📊 Silver Rows:", silver_df.count())

# -----------------------------
# METRICS (BUSINESS LOGIC)
# -----------------------------

# Total Revenue
revenue_df = silver_df.groupBy() \
    .agg(sum("payment_value").alias("total_revenue"))

# Total Orders
orders_df = silver_df.select("order_id").distinct() \
    .agg(count("order_id").alias("total_orders"))

# Average Order Value
aov_df = silver_df.groupBy() \
    .agg(avg("payment_value").alias("avg_order_value"))

# Orders by Status
status_df = silver_df.groupBy("order_status") \
    .agg(count("order_id").alias("order_count"))

print("📊 Metrics calculated")

# -----------------------------
# WRITE GOLD
# -----------------------------
revenue_df.write.format("delta").mode("overwrite").save(gold_base + "revenue/")
orders_df.write.format("delta").mode("overwrite").save(gold_base + "orders/")
aov_df.write.format("delta").mode("overwrite").save(gold_base + "aov/")
status_df.write.format("delta").mode("overwrite").save(gold_base + "order_status/")

print("✅ Gold Layer Completed")

# -----------------------------
# DISPLAY RESULTS
# -----------------------------
display(revenue_df)
display(orders_df)
display(aov_df)
display(status_df)

In [0]:
display(dbutils.fs.ls("abfss://gold@vasudatalake1.dfs.core.windows.net/dev/"))

In [0]:
display(spark.read.format("delta").load("abfss://gold@vasudatalake1.dfs.core.windows.net/dev/order_status/"))

In [0]:
display(dbutils.fs.ls("abfss://gold@vasudatalake1.dfs.core.windows.net/dev/revenue/"))

In [0]:
display(
    spark.read.format("delta")
    .load("abfss://gold@vasudatalake1.dfs.core.windows.net/dev/revenue/")
)

# BEFORE OPTIMIZATION

In [0]:
import time
from pyspark.sql.functions import col

# Analyze execution plan BEFORE optimization
print("📊 BEFORE OPTIMIZATION - Execution Plan")
before_opt_start = time.time()

# Sample query to measure
sample_query_df = spark.read.format("delta").load(silver_base + "ecommerce/") \
    .filter(col("order_status") == "delivered") \
    .filter(col("payment_type") == "credit_card") \
    .select("order_id", "customer_id", "payment_value")

# Show execution plan
sample_query_df.explain(mode="extended")

# Execute and time
sample_query_df.collect()
before_opt_end = time.time()
before_optimization_runtime = before_opt_end - before_opt_start
print(f"⏱️ Before Optimization Runtime: {before_optimization_runtime:.4f} seconds")

# AFTER OPTIMIZATION

In [0]:
# Run OPTIMIZE on Silver table
print("🔧 Running OPTIMIZE on Silver table...")
optimize_start = time.time()

spark.sql(f"OPTIMIZE delta.`{silver_base}ecommerce/`")

optimize_end = time.time()
print(f"✅ OPTIMIZE completed in {optimize_end - optimize_start:.2f} seconds")

# Apply Z-Ordering

In [0]:
# Apply Z-Ordering on frequently queried columns
print("🔧 Applying Z-Ordering on Silver table (order_status, payment_type, customer_id)...")
zorder_start = time.time()

spark.sql(f"""
    OPTIMIZE delta.`{silver_base}ecommerce/`
    ZORDER BY (order_status, payment_type, customer_id)
""")

zorder_end = time.time()
print(f"✅ Z-Ordering completed in {zorder_end - zorder_start:.2f} seconds")

# Cache Usage

In [0]:
# Cache frequently accessed data
print("🔧 Caching Silver table...")
cache_start = time.time()

silver_cached_df = spark.read.format("delta").load(silver_base + "ecommerce/")
silver_cached_df.cache()
silver_cached_df.count()  # Trigger cache

cache_end = time.time()
print(f"✅ Cache completed in {cache_end - cache_start:.2f} seconds")

# After Optimization 

In [0]:
# Analyze execution plan AFTER optimization
print("📊 AFTER OPTIMIZATION - Execution Plan")
after_opt_start = time.time()

# Same query after optimization
optimized_query_df = silver_cached_df \
    .filter(col("order_status") == "delivered") \
    .filter(col("payment_type") == "credit_card") \
    .select("order_id", "customer_id", "payment_value")

# Show execution plan
optimized_query_df.explain(mode="extended")

# Execute and time
optimized_query_df.collect()
after_opt_end = time.time()
after_optimization_runtime = after_opt_end - after_opt_start
print(f"⏱️ After Optimization Runtime: {after_optimization_runtime:.4f} seconds")